In [1]:
import os
import glob
import requests
import re
import pandas as pd
pd.set_option('max_colwidth', 400)  #wider pandas columns upon render
import numpy as np
from bs4 import BeautifulSoup
from itertools import islice
from thefuzz import fuzz, process
from datetime import date

os.getcwd()

'c:\\Users\\TooFastDan\\OneDrive - Baylor College of Medicine\\APSA\\2024-07-29_Texas-STAR'

In [3]:
def parse_dox(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        soup = BeautifulSoup(file, "html.parser")
    return soup

In [4]:
def residency_parser(file):
    # Beautiful Soup
    soup = parse_dox(filename=file)
    
    # Extracting the residency list div from the page
    res_list = soup.find("div", class_ = "residency-program-list")
    
    # Program names
    programs = res_list.find_all("a", class_ = "residency-result-program-title")
    programs = [p.text for p in programs]

    # Program locations
    locations = res_list.find_all("p", class_ = "residency-result-location")
    locations = [l.text for l in locations]

    # Program Sizes
    program_size = res_list.find_all("p", class_ = "residency-result-stats-program-size")
    program_size = [ps.text for ps in program_size]
    pattern = r'\d+'
    program_size = [int(re.findall(pattern, ps)[0]) for ps in program_size]

    # Program ranks
    ranks = np.arange(1, len(programs) + 1, 1)
    
    # Making a dataframe
    df = pd.DataFrame({"program": programs, "location": locations, "program_size": program_size, "rank": ranks})
    return df

In [5]:
# Getting all .html files from residency rankings from doximity
html_list = glob.glob("dox_rankings/*.html")
html_list[10:13]

['dox_rankings\\Internal Medicine-Pediatrics_research-output.html',
 'dox_rankings\\Internal Medicine_reputation.html',
 'dox_rankings\\Internal Medicine_research-output.html']

In [6]:
# Looping over each file to get rankings
df_list = []
for h in html_list:
    # Getting specialty and sort metrics from the html file names
    file_name = h.replace("dox_rankings\\", "").replace(".html", "")
    specialty, sort_by = file_name.split("_")
    
    # Beatiful Soup parsing
    print(specialty, sort_by)
    df = residency_parser(file=h)
    df["specialty"] = specialty
    df["sort_by"] = sort_by
    df_list.append(df)
    
# Concatenating things to a single dataframe
df_complete = pd.concat(df_list)
df_complete

Anesthesiology reputation
Anesthesiology research-output
Child Neurology reputation
Child Neurology research-output
Dermatology reputation
Dermatology research-output
Emergency Medicine reputation
Emergency Medicine research-output
Family Medicine reputation
Internal Medicine-Pediatrics reputation
Internal Medicine-Pediatrics research-output
Internal Medicine reputation
Internal Medicine research-output
Neurological Surgery reputation
Neurological Surgery research-output
Neurology reputation
Neurology research-output
Nuclear Medicine reputation
Nuclear Medicine research-output
Obstetrics and Gynecology reputation
Obstetrics and Gynecology research-output
Occupational Medicine reputation
Occupational Medicine research-output
Ophthalmology reputation
Ophthalmology research-output
Orthopaedic Surgery reputation
Orthopaedic Surgery research-output
Otolaryngology reputation
Otolaryngology research-output
Pathology reputation
Pathology research-output
Pediatrics-Medical Genetics size-of-prog

,program,location,program_size,rank,specialty,sort_by
0,Mass General Brigham/Massachusetts General Hos...,Boston,116,1,Anesthesiology,reputation
1,University of California (San Francisco),San Francisco,96,2,Anesthesiology,reputation
2,Stanford Health Care-Sponsored Stanford Univer...,Stanford,108,3,Anesthesiology,reputation
3,Mass General Brigham/Brigham and Women's Hospital,Boston,128,4,Anesthesiology,reputation
4,Duke University Hospital,Durham,60,5,Anesthesiology,reputation
...,...,...,...,...,...,...
72,Wake Forest University Baptist Medical Center,Winston Salem,5,73,Vascular Surgery,size-of-program
73,Washington University/B-JH/SLCH Consortium,Saint Louis,5,74,Vascular Surgery,size-of-program
74,Yale-New Haven Medical Center,New Haven,5,75,Vascular Surgery,size-of-program
75,Zucker School of Medicine at Hofstra/Northwell,New Hyde Park,5,76,Vascular Surgery,size-of-program


In [7]:
df_complete[["specialty", "sort_by"]].value_counts()

specialty                             sort_by        
Family Medicine                       reputation         766
Internal Medicine                     research-output    639
                                      reputation         639
Surgery                               research-output    358
                                      reputation         358
Psychiatry                            reputation         317
                                      research-output    317
Obstetrics and Gynecology             research-output    294
                                      reputation         294
Emergency Medicine                    research-output    286
                                      reputation         286
Pediatrics                            research-output    218
                                      reputation         218
Orthopaedic Surgery                   reputation         207
                                      research-output    207
Radiology-Diagnostic           

In [ ]:
# Exporting to excel doc
#df_complete.to_excel("tables/doximity_rankings.xlsx", index=False)

In [59]:
soup = parse_dox(filename="dox_rankings/Internal Medicine_research-output.html")
#print(soup)

In [60]:
# Extracting the residency list div from the page
res_list = soup.find("div", class_ = "residency-program-list")

In [61]:
# Program names
programs = res_list.find_all("a", class_ = "residency-result-program-title")
programs = [p.text for p in programs]

# Program locations
locations = res_list.find_all("p", class_ = "residency-result-location")
locations = [l.text for l in locations]

# Program Sizes
program_size = res_list.find_all("p", class_ = "residency-result-stats-program-size")
program_size = [ps.text for ps in program_size]
pattern = r'\d+'
program_size = [int(re.findall(pattern, ps)[0]) for ps in program_size]

# Program ranks
ranks = np.arange(1, len(programs) + 1, 1)

In [62]:
# Making a dataframe
df = pd.DataFrame({"program": programs, "location": locations, "program_size": program_size, "rank": ranks})
df["specialty"] = "Anesthesiology"
df["sort_by"] = "reputation"
df

,program,location,program_size,rank,specialty,sort_by
0,Mass General Brigham/Massachusetts General Hos...,Boston,168,1,Anesthesiology,reputation
1,Mass General Brigham/Brigham and Women's Hospital,Boston,174,2,Anesthesiology,reputation
2,University of California (San Francisco),San Francisco,185,3,Anesthesiology,reputation
3,University of Pennsylvania Health System,Philadelphia,182,4,Anesthesiology,reputation
4,Johns Hopkins University,Baltimore,145,5,Anesthesiology,reputation
...,...,...,...,...,...,...
634,WakeMed Health and Hospitals,Raleigh,15,635,Anesthesiology,reputation
635,Western Reserve Hospital,Cuyahoga Falls,18,636,Anesthesiology,reputation
636,Willis-Knighton Health System,Shreveport,30,637,Anesthesiology,reputation
637,Zucker School of Medicine at Hofstra/Northwell...,Mount Kisco,32,638,Anesthesiology,reputation


## **Fuzzy string match for inexact program matches**
#### Imported from R

In [41]:
star = pd.read_csv("tables/texasSTAR_unique_programs.csv")
unmatched = pd.read_csv("tables/unmatched_unique_programs.csv")
dox = pd.read_excel("tables/doximity_rankings.xlsx")

display(star.head())
display(unmatched.head())
display(dox.head())

,star_programs,program,specialty
0,Baylor College of Medicine_Anesthesiology,Baylor College of Medicine,Anesthesiology
1,Beth Israel Deaconess Medical Center/Harvard Medical School_Anesthesiology,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology
2,Duke University Hospital_Anesthesiology,Duke University Hospital,Anesthesiology
3,Massachusetts General Hospital/Harvard Medical School_Anesthesiology,Massachusetts General Hospital/Harvard Medical School,Anesthesiology
4,Texas A&M College of Medicine-Scott and White_Anesthesiology,Texas A&M College of Medicine-Scott and White,Anesthesiology


,unmatched_programs,program,specialty
0,Beth Israel Deaconess Medical Center/Harvard Medical School_Anesthesiology,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology
1,Massachusetts General Hospital/Harvard Medical School_Anesthesiology,Massachusetts General Hospital/Harvard Medical School,Anesthesiology
2,Texas A&M College of Medicine-Scott and White_Anesthesiology,Texas A&M College of Medicine-Scott and White,Anesthesiology
3,University of Texas Health Science Center School of Medicine at San Antonio_Anesthesiology,University of Texas Health Science Center School of Medicine at San Antonio,Anesthesiology
4,University of Texas Southwestern Medical School_Anesthesiology,University of Texas Southwestern Medical School,Anesthesiology


,program,location,program_size,rank,specialty,sort_by
0,Mass General Brigham/Massachusetts General Hospital,Boston,116,1,Anesthesiology,reputation
1,University of California (San Francisco),San Francisco,96,2,Anesthesiology,reputation
2,Stanford Health Care-Sponsored Stanford University,Stanford,108,3,Anesthesiology,reputation
3,Mass General Brigham/Brigham and Women's Hospital,Boston,128,4,Anesthesiology,reputation
4,Duke University Hospital,Durham,60,5,Anesthesiology,reputation


In [31]:
for index, row in islice(unmatched.iterrows(), 10):
    program = row["program"]
    specialty = row["specialty"]
    print(program, specialty)

Beth Israel Deaconess Medical Center/Harvard Medical School Anesthesiology
Massachusetts General Hospital/Harvard Medical School Anesthesiology
Texas A&M College of Medicine-Scott and White Anesthesiology
University of Texas Health Science Center School of Medicine at San Antonio Anesthesiology
University of Texas Southwestern Medical School Anesthesiology
Brigham and Women's Hospital/Harvard Medical School Anesthesiology
Icahn School of Medicine at Mount Sinai Anesthesiology
Icahn School of Medicine at Mount Sinai (St Luke's-Roosevelt Hospital Center) Anesthesiology
Loyola University Anesthesiology
New York University School of Medicine Anesthesiology


In [51]:
match_list = []
for index, row in islice(unmatched.iterrows(), 5):
    program = row["program"]
    specialty = row["specialty"]
    
    dox_filt = dox[dox["specialty"]==specialty]
    dox_filt = dox_filt["program"].unique()
    
    ratio = []
    partial_ratio = []
    sort_ratio = []
    set_ratio = []
    partial_token = []
    dox_program_list = []
    for dox_program in dox_filt:
        ratio.append(fuzz.ratio(program, dox_program))
        partial_ratio.append(fuzz.partial_ratio(program, dox_program))
        sort_ratio.append(fuzz.token_sort_ratio(program, dox_program))
        set_ratio.append(fuzz.token_set_ratio(program, dox_program))
        partial_token.append(fuzz.partial_token_sort_ratio(program, dox_program))
        dox_program_list.append(dox_program)
    
    df = pd.DataFrame({"ratio": ratio, "partial_ratio": partial_ratio, "sort_ratio": sort_ratio, "set_ratio": set_ratio, "partial_token": partial_token, "dox_program": dox_program_list})
    df["star_program"] = program
    df["specialty"] = specialty
    df["mean_ratio"] = df.mean(axis=1)
    df = df.sort_values("mean_ratio", ascending=False).reset_index(drop=True)
    display(df.head(10))
    
    selection = input("Select the top match by index: ")
    if selection=="done":
        print("finished with looping")
        break
    try:
        selection = int(selection)
        matched_df = pd.DataFrame(df.iloc[selection]).transpose()
        match_list.append(matched_df)
        display(matched_df)
    except:
        print("unable to match")
        unmatched_df = pd.DataFrame({"ratio": np.nan, "partial_ratio": np.nan, "sort_ratio": np.nan, "set_ratio": np.nan, "partial_token": np.nan, "dox_program": program, "star_program": program, "specialty": specialty, "mean_ratio": "unable to match"}, index=[0])
        match_list.append(unmatched_df)
        display(unmatched_df)

df_total = pd.concat(match_list)
display(df_total)

C:\Users\TooFastDan\AppData\Local\Temp\ipykernel_19540\696497551.py:26: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df["mean_ratio"] = df.mean(axis=1)


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,76,100,76,100,78,Beth Israel Deaconess Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,86.0
1,50,75,43,72,79,UMass Chan Medical School,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,63.8
2,46,85,46,82,60,Maine Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,63.8
3,43,80,41,82,60,Tufts Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,61.2
4,48,80,48,72,57,Maimonides Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,61.0
5,45,76,40,80,61,Albany Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,60.4
6,47,73,45,70,59,Westchester Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,58.8
7,47,73,49,70,54,St Joseph's Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,58.6
8,51,64,51,62,60,Rutgers Health/New Jersey Medical School,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,57.6
9,45,69,52,65,55,Yale-New Haven Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,57.2


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,76,100,76,100,78,Beth Israel Deaconess Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,86.0


C:\Users\TooFastDan\AppData\Local\Temp\ipykernel_19540\696497551.py:26: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df["mean_ratio"] = df.mean(axis=1)


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,58,74,63,82,74,Mass General Brigham/Massachusetts General Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,70.2
1,54,75,44,72,75,UMass Chan Medical School,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,64.0
2,54,61,47,54,61,Rutgers Health/New Jersey Medical School,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,55.4
3,47,53,51,62,58,Mass General Brigham/Brigham and Women's Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,54.2
4,51,56,45,53,54,Rutgers Health/Robert Wood Johnson Medical School,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,51.8
5,48,51,45,54,58,Case Western Reserve University/University Hospitals Cleveland Medical Center,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,51.2
6,33,63,36,59,63,Henry Ford Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,50.8
7,37,64,37,53,59,Robert Packer Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,50.0
8,39,62,39,50,58,Duke University Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,49.6
9,40,46,48,48,65,Methodist Hospital (Houston),Massachusetts General Hospital/Harvard Medical School,Anesthesiology,49.4


...

,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,69,83,83,98,79,University of Texas Health Science Center San Antonio Joe and Teresa Lozano Long School of Medicine,University of Texas Health Science Center School of Medicine at San Antonio,Anesthesiology,82.4


C:\Users\TooFastDan\AppData\Local\Temp\ipykernel_19540\696497551.py:26: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df["mean_ratio"] = df.mean(axis=1)


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,87,93,85,92,92,University of Texas Southwestern Medical Center,University of Texas Southwestern Medical School,Anesthesiology,89.8
1,57,89,51,79,76,University of Toledo,University of Texas Southwestern Medical School,Anesthesiology,70.4
2,60,77,60,72,70,University of Rochester,University of Texas Southwestern Medical School,Anesthesiology,67.8
3,70,68,59,76,61,University of Texas Medical Branch Hospitals,University of Texas Southwestern Medical School,Anesthesiology,66.8
4,58,78,53,72,73,UMass Chan Medical School,University of Texas Southwestern Medical School,Anesthesiology,66.8
5,56,80,50,76,71,University of Chicago,University of Texas Southwestern Medical School,Anesthesiology,66.6
6,53,80,47,76,71,University of Florida,University of Texas Southwestern Medical School,Anesthesiology,65.4
7,52,78,52,74,69,University of Michigan,University of Texas Southwestern Medical School,Anesthesiology,65.0
8,52,78,49,74,69,University of Colorado,University of Texas Southwestern Medical School,Anesthesiology,64.4
9,52,78,49,74,69,University of Maryland,University of Texas Southwestern Medical School,Anesthesiology,64.4


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
1,57,89,51,79,76,University of Toledo,University of Texas Southwestern Medical School,Anesthesiology,70.4


,ratio,partial_ratio,sort_ratio,set_ratio,partial_token,dox_program,star_program,specialty,mean_ratio
0,76,100,76,100,78,Beth Israel Deaconess Medical Center,Beth Israel Deaconess Medical Center/Harvard Medical School,Anesthesiology,86.0
0,58,74,63,82,74,Mass General Brigham/Massachusetts General Hospital,Massachusetts General Hospital/Harvard Medical School,Anesthesiology,70.2
0,NaN,NaN,NaN,NaN,NaN,Texas A&M College of Medicine-Scott and White,Texas A&M College of Medicine-Scott and White,Anesthesiology,unable to match
0,69,83,83,98,79,University of Texas Health Science Center San Antonio Joe and Teresa Lozano Long School of Medicine,University of Texas Health Science Center School of Medicine at San Antonio,Anesthesiology,82.4
1,57,89,51,79,76,University of Toledo,University of Texas Southwestern Medical School,Anesthesiology,70.4


In [12]:
ratio = []
for index, row in star.iterrows():
    r = fuzz.ratio(row["star_programs"], row["program"])
    ratio.append(r)
star["ratio"] = ratio
star

,star_programs,program,specialty,ratio
0,Baylor College of Medicine_Anesthesiology,Baylor College of Medicine,Anesthesiology,78
1,Beth Israel Deaconess Medical Center/Harvard M...,Beth Israel Deaconess Medical Center/Harvard M...,Anesthesiology,89
2,Duke University Hospital_Anesthesiology,Duke University Hospital,Anesthesiology,76
3,Massachusetts General Hospital/Harvard Medical...,Massachusetts General Hospital/Harvard Medical...,Anesthesiology,88
4,Texas A&M College of Medicine-Scott and White_...,Texas A&M College of Medicine-Scott and White,Anesthesiology,86
...,...,...,...,...
10729,Loma Linda University Health Education Consort...,Loma Linda University Health Education Consort...,Preventive Medicine,90
10730,Texas Department of State Health Services Publ...,Texas Department of State Health Services Publ...,Preventive Medicine,90
10731,University at Buffalo Public Health and Genera...,University at Buffalo Public Health and Genera...,Preventive Medicine,87
10732,University of Texas Health Science Center at H...,University of Texas Health Science Center at H...,Preventive Medicine,88


In [56]:
today = str(date.today())
print(today + "_Anesthiology")

2024-12-12_Anesthiology


## **AAMC Table B4 Parser**

Source: https://www.aamc.org/data-reports/students-residents/data/report-residents/2024/table-b4-md-phd-residents-gme-specialty

In [13]:
b4 = pd.read_excel("AAMC_data/AAMC_Table-B4.xlsx", sheet_name="table_b4")
b4 = b4.set_index("Specialties")
b4.head()

,MD-PhD First-Year Residents 2016,MD-PhD Active Residents 2016,Total Active Residents 2016,MD-PhD First-Year Residents 2017,MD-PhD Active Residents 2017,Total Active Residents 2017,MD-PhD First-Year Residents 2018,MD-PhD Active Residents 2018,Total Active Residents 2018,MD-PhD First-Year Residents 2019,...,Total Active Residents 2020,MD-PhD First-Year Residents 2021,MD-PhD Active Residents 2021,Total Active Residents 2021,MD-PhD First-Year Residents 2022,MD-PhD Active Residents 2022,Total Active Residents 2022,MD-PhD First-Year Residents 2023,MD-PhD Active Residents 2023,Total Active Residents 2023
Specialties,,,,,,,,,,,,,,,,,,,,,
Aerospace Medicine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,1.0,7.0,0.0,2.0,5.0,0.0,1.0,4.0
Allergy and Immunology,0.0,15.0,212.0,0.0,16.0,221.0,0.0,17.0,221.0,1.0,...,224.0,0.0,20.0,236.0,0.0,22.0,240.0,0.0,22.0,228.0
Anesthesiology,16.0,90.0,4487.0,17.0,89.0,4444.0,14.0,81.0,4491.0,24.0,...,4441.0,14.0,90.0,4795.0,18.0,94.0,4969.0,14.0,92.0,5209.0
Adult Cardiothoracic Anesthesiology (Anesthesiology),0.0,5.0,145.0,0.0,0.0,136.0,0.0,2.0,134.0,0.0,...,147.0,0.0,6.0,141.0,0.0,3.0,149.0,0.0,2.0,135.0
Critical Care Medicine (Anesthesiology),0.0,3.0,136.0,0.0,7.0,131.0,0.0,6.0,126.0,0.0,...,130.0,0.0,5.0,140.0,0.0,3.0,142.0,0.0,5.0,127.0


In [16]:
# Formatting to long format per year
year_list = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]
b4_list = []
for year in year_list:
    b4_filt = b4.filter(like=year, axis=1).reset_index()
    b4_filt.columns = ["specialties", "mdphd_firstyear", "mdphd_active", "total_active"]
    b4_filt["year"] = int(year)
    b4_list.append(b4_filt)
b4_complete = pd.concat(b4_list)
b4_complete

,specialties,mdphd_firstyear,mdphd_active,total_active,year
0,Aerospace Medicine,NaN,NaN,NaN,2016
1,Allergy and Immunology,0.0,15.0,212.0,2016
2,Anesthesiology,16.0,90.0,4487.0,2016
3,Adult Cardiothoracic Anesthesiology (Anesthesiology),0.0,5.0,145.0,2016
4,Critical Care Medicine (Anesthesiology),0.0,3.0,136.0,2016
...,...,...,...,...,...
160,Pediatrics/Physical Medicine and Rehabilitation,0.0,1.0,11.0,2023
161,Pediatrics/Psychiatry/Child and Adolescent Psychiatry,0.0,2.0,97.0,2023
162,Psychiatry/Family Practice,0.0,1.0,49.0,2023
163,Psychiatry/Neurology,NaN,NaN,NaN,2023


In [15]:
# Exporting to excel
#b4_complete.to_excel("AAMC_data/AAMC_Table-B4_processed.xlsx", index=False)